# CLEF 2026 Task 3 — Hybrid Model Backtest

Backtest the live endpoint logic on the official training set (`TheFinAI/CLEF_Task3_Trading`) and tune thresholds + weights against a Cumulative-Return-heavy + Sharpe blended objective.

Signals used (no lookahead):
- **FinBERT** sentiment on news
- **Filing alignment** (TSLA only): chunk 10-K / 10-Q, embed with MiniLM, cosine-similarity-weighted news sentiment + filing baseline tone
- **Price trend**: blended 3-day and 7-day return, reconstructed from per-asset price series in the dataset
- **Organizer momentum** label
- **yfinance peers/macro/ratios** as of each historical date

Output: best parameter set per asset, plus updated thresholds to drop into `app.py`.

## 1. Install

In [ ]:
!pip install -q transformers torch datasets pandas numpy matplotlib yfinance sentence-transformers scikit-learn

## 2. Load dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('TheFinAI/CLEF_Task3_Trading')
btc_rows = list(ds['BTC'])
tsla_rows = list(ds['TSLA'])
print('BTC rows :', len(btc_rows))
print('TSLA rows:', len(tsla_rows))
print('Fields    :', list(btc_rows[0].keys()))

In [ ]:
# Build per-asset DataFrame sorted by date so we can reconstruct history_price for the trend signal
def to_df(rows):
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['prices'] = pd.to_numeric(df['prices'], errors='coerce')
    return df

btc_df  = to_df(btc_rows)
tsla_df = to_df(tsla_rows)
print(btc_df[['date', 'prices', 'momentum']].head())
print(tsla_df[['date', 'prices', 'momentum']].head())

## 3. Load models (FinBERT + MiniLM)

In [ ]:
from transformers import pipeline, AutoTokenizer
from sentence_transformers import SentenceTransformer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
finbert = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    tokenizer=tokenizer,
    top_k=None,
    truncation=True,
    max_length=512,
    device=-1,
)
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Models loaded.')

## 4. Helpers — sentiment, chunking, embedding

In [ ]:
def chunk_text(text, max_tokens=510):
    if not isinstance(text, str) or not text.strip():
        return []
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    out = []
    for i in range(0, len(ids), max_tokens):
        ch = tokenizer.decode(ids[i:i + max_tokens], skip_special_tokens=True)
        if ch.strip():
            out.append(ch)
    return out

def finbert_score(text):
    chunks = chunk_text(text)
    if not chunks:
        return 0.0
    vals = []
    for ch in chunks:
        labels = {x['label'].lower(): x['score'] for x in finbert(ch)[0]}
        vals.append(labels.get('positive', 0.0) - labels.get('negative', 0.0))
    return float(np.mean(vals))

def embed_texts(texts):
    if not texts:
        return np.zeros((0, 384), dtype=np.float32)
    return embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

## 5. Filing-alignment signals (TSLA only)

For each row with a filing, build:
- `filing_chunks` (list of paragraph chunks)
- `filing_chunk_sentiments` (FinBERT sentiment per chunk)
- `filing_chunk_embeds` (MiniLM embeddings per chunk)

Per news item, derive:
- `materiality` = max cosine sim between news embedding and filing chunk embeddings
- `weighted_news_sentiment` = Σ(materiality × news_sentiment) / Σ(materiality)
- `filing_baseline_tone` = mean filing chunk sentiment
- `disagreement` = filing_baseline_tone - weighted_news_sentiment

In [ ]:
def gather_filing_text(row):
    parts = []
    for key in ('10k', '10q'):
        v = row.get(key)
        if v is None:
            continue
        if isinstance(v, list):
            parts.extend([str(x) for x in v if x])
        elif isinstance(v, str) and v.strip():
            parts.append(v)
    return parts

def split_filing_paragraphs(filing_strings, max_chars=1500):
    paras = []
    for s in filing_strings:
        for chunk in s.split('\n\n'):
            chunk = chunk.strip()
            if len(chunk) < 80:
                continue
            for i in range(0, len(chunk), max_chars):
                paras.append(chunk[i:i + max_chars])
    return paras

In [ ]:
# Filing cache: maps a filing-content hash -> precomputed (sentiments, embeddings)
import hashlib
_filing_cache = {}

def get_filing_signals(filing_strings):
    if not filing_strings:
        return None
    h = hashlib.md5(('||'.join(filing_strings)).encode('utf-8')).hexdigest()
    if h in _filing_cache:
        return _filing_cache[h]
    paras = split_filing_paragraphs(filing_strings)
    if not paras:
        _filing_cache[h] = None
        return None
    # cap chunks for speed
    paras = paras[:60]
    sentiments = np.array([finbert_score(p) for p in paras])
    embeds = embed_texts(paras)
    out = {'sentiments': sentiments, 'embeds': embeds, 'baseline_tone': float(sentiments.mean())}
    _filing_cache[h] = out
    return out

## 6. yfinance enrichment (peers, sector, macro, ratios)

Pull historical OHLC for peer tickers once, then for each row look up returns as-of that date. Strict no-lookahead: only data available at or before the row date is used.

In [ ]:
import yfinance as yf

PEERS = {
    'TSLA': ['RIVN', 'GM', 'F', 'NIO', 'LCID'],
    'BTC':  ['ETH-USD', 'COIN'],
}
SECTOR_PROXY = {'TSLA': 'XLY', 'BTC': 'COIN'}
MACRO_TICKERS = ['^VIX', 'DX-Y.NYB', '^TNX']

all_tickers = set(MACRO_TICKERS)
for arr in PEERS.values():
    all_tickers.update(arr)
for v in SECTOR_PROXY.values():
    all_tickers.add(v)
all_tickers.update(['TSLA', 'BTC-USD'])

px_start = (btc_df['date'].min() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
px_end   = (btc_df['date'].max() + pd.Timedelta(days=2)).strftime('%Y-%m-%d')
print('Pulling yfinance', px_start, '->', px_end, 'for', len(all_tickers), 'tickers')
px = yf.download(list(all_tickers), start=px_start, end=px_end, auto_adjust=True, progress=False)['Close']
px.index = pd.to_datetime(px.index)
print('Shape:', px.shape)
px.tail()

In [ ]:
def safe_pct(series, asof, lookback_days):
    sub = series.loc[:asof].dropna()
    if len(sub) < 2:
        return 0.0
    last = sub.iloc[-1]
    target_date = asof - pd.Timedelta(days=lookback_days)
    prior_sub = sub.loc[:target_date]
    if len(prior_sub) == 0:
        return 0.0
    prior = prior_sub.iloc[-1]
    return float((last - prior) / prior) if prior > 0 else 0.0

def peer_signal(asset, asof):
    rets = []
    for t in PEERS.get(asset, []):
        if t in px.columns:
            rets.append(safe_pct(px[t], asof, 5))
    return float(np.mean(rets)) if rets else 0.0

def sector_signal(asset, asof):
    t = SECTOR_PROXY.get(asset)
    if t and t in px.columns:
        return safe_pct(px[t], asof, 5)
    return 0.0

def macro_signal(asof):
    # negative VIX change = risk on; positive = risk off
    vix = -safe_pct(px['^VIX'], asof, 5) if '^VIX' in px.columns else 0.0
    return float(np.clip(vix, -0.10, 0.10))

## 7. Compute features for every row (the slow step — 30–45 min on CPU)

In [ ]:
from tqdm import tqdm

def reconstruct_price_history(asset_df, idx, lookback=10):
    # Use rows BEFORE idx only -> no lookahead
    return asset_df.iloc[max(0, idx - lookback):idx][['date', 'prices']].rename(columns={'prices': 'price'}).to_dict('records')

def price_trend_from_history(history):
    if not history or len(history) < 2:
        return 0.0
    h = sorted(history, key=lambda x: x['date'])
    last = float(h[-1]['price'])
    r3 = (last - float(h[-min(3, len(h))]['price'])) / float(h[-min(3, len(h))]['price']) if h[-min(3, len(h))]['price'] else 0
    r7 = (last - float(h[-min(7, len(h))]['price'])) / float(h[-min(7, len(h))]['price']) if h[-min(7, len(h))]['price'] else 0
    return max(-0.15, min(0.15, 0.5 * r3 + 0.5 * r7))

def get_news_list(row):
    n = row.get('news') or []
    if isinstance(n, list):
        return [x for x in n if isinstance(x, str) and x.strip()]
    if isinstance(n, str) and n.strip():
        return [n]
    return []

def features_for_row(asset_df, idx, asset):
    row = asset_df.iloc[idx].to_dict()
    asof = pd.to_datetime(row['date'])
    news = get_news_list(row)
    news_sents = np.array([finbert_score(n) for n in news[:10]]) if news else np.array([0.0])
    raw_news_sentiment = float(news_sents.mean())

    # Filing alignment (TSLA only)
    weighted_news_sent = raw_news_sentiment
    filing_tone = 0.0
    disagreement = 0.0
    if asset == 'TSLA':
        filings = gather_filing_text(row)
        sig = get_filing_signals(filings)
        if sig and news:
            news_emb = embed_texts(news[:10])
            sims = news_emb @ sig['embeds'].T  # (n_news, n_chunks)
            mat  = sims.max(axis=1).clip(0)    # materiality per news
            if mat.sum() > 1e-6:
                weighted_news_sent = float((mat * news_sents) .sum() / mat.sum())
            filing_tone = sig['baseline_tone']
            disagreement = filing_tone - weighted_news_sent

    # Price trend (no lookahead)
    history = reconstruct_price_history(asset_df, idx)
    trend = price_trend_from_history(history)

    # Organizer momentum label
    mom = str(row.get('momentum') or 'neutral').lower()
    mom_adj = 0.05 if mom == 'bullish' else (-0.05 if mom == 'bearish' else 0.0)

    # External signals
    peer = peer_signal(asset, asof)
    sector = sector_signal(asset, asof)
    macro = macro_signal(asof)

    return {
        'date': row['date'], 'asset': asset, 'price': row['prices'],
        'future_price_diff': row.get('future_price_diff'),
        'raw_news_sent':       raw_news_sentiment,
        'weighted_news_sent':  weighted_news_sent,
        'filing_tone':         filing_tone,
        'disagreement':        disagreement,
        'price_trend':         trend,
        'mom_adj':             mom_adj,
        'peer':                peer,
        'sector':              sector,
        'macro':               macro,
    }

In [ ]:
btc_feats = []
for i in tqdm(range(len(btc_df)), desc='BTC features'):
    btc_feats.append(features_for_row(btc_df, i, 'BTC'))
btc_feats = pd.DataFrame(btc_feats)
btc_feats.head()

In [ ]:
tsla_feats = []
for i in tqdm(range(len(tsla_df)), desc='TSLA features'):
    tsla_feats.append(features_for_row(tsla_df, i, 'TSLA'))
tsla_feats = pd.DataFrame(tsla_feats)
tsla_feats.head()

In [ ]:
btc_feats.to_parquet('btc_features.parquet')
tsla_feats.to_parquet('tsla_features.parquet')
print('Features cached.')

## 8. Score function and backtest engine

In [ ]:
def combine_score(row, weights, asset):
    if asset == 'BTC':
        return (
            weights['w_news']   * row['raw_news_sent']
          + weights['w_trend']  * row['price_trend']
          + weights['w_mom']    * row['mom_adj']
          + weights['w_peer']   * row['peer']
          + weights['w_sector'] * row['sector']
          + weights['w_macro']  * row['macro']
        )
    # TSLA — use weighted news sent + filing alignment
    return (
        weights['w_news']    * row['weighted_news_sent']
      + weights['w_filing']  * row['filing_tone']
      + weights['w_disagree']* (-row['disagreement'])
      + weights['w_trend']   * row['price_trend']
      + weights['w_mom']     * row['mom_adj']
      + weights['w_peer']    * row['peer']
      + weights['w_sector']  * row['sector']
      + weights['w_macro']   * row['macro']
    )

def actions_from_scores(scores, buy_th, sell_th):
    out = np.where(scores >= buy_th, 1, np.where(scores <= sell_th, -1, 0))
    return out

def backtest_metrics(feats, weights, buy_th, sell_th):
    f = feats.sort_values('date').reset_index(drop=True).copy()
    f['score']    = f.apply(lambda r: combine_score(r, weights, r['asset']), axis=1)
    f['position'] = actions_from_scores(f['score'].values, buy_th, sell_th)
    f['ret']      = f['price'].pct_change().fillna(0)
    f['pos_lag']  = f['position'].shift(1).fillna(0)
    f['strat_ret']= f['pos_lag'] * f['ret']

    cum    = (1 + f['strat_ret']).cumprod()
    bh_cum = (1 + f['ret']).cumprod()
    cr     = float(cum.iloc[-1] - 1) if len(cum) else 0.0
    bh_cr  = float(bh_cum.iloc[-1] - 1) if len(bh_cum) else 0.0
    sharpe = float(f['strat_ret'].mean() / (f['strat_ret'].std() + 1e-9) * np.sqrt(252))
    mdd    = float(((cum / cum.cummax()) - 1).min()) if len(cum) else 0.0
    return {'CR': cr, 'BH_CR': bh_cr, 'Sharpe': sharpe, 'MaxDD': mdd, 'final': float(cum.iloc[-1]) if len(cum) else 1.0}

def objective(metrics):
    # heavy on return: 0.7 normalized CR + 0.3 normalized Sharpe, MDD softly penalized
    cr_score = np.tanh(metrics['CR'] * 2)            # squashes to (-1, 1), CR=0.5 -> ~0.76
    sh_score = np.tanh(metrics['Sharpe'] / 2)        # Sharpe=2 -> ~0.76
    mdd_pen  = max(0.0, -metrics['MaxDD'] - 0.30) * 0.5  # only penalize drawdowns past -30%
    return 0.7 * cr_score + 0.3 * sh_score - mdd_pen

## 9. Grid search per asset

In [ ]:
import itertools

def grid_search_btc(feats):
    grid = {
        'w_news':   [0.6, 0.8, 1.0, 1.2],
        'w_trend':  [0.3, 0.5, 0.7, 1.0],
        'w_mom':    [0.5, 1.0, 1.5],
        'w_peer':   [0.0, 0.5, 1.0],
        'w_sector': [0.0, 0.5],
        'w_macro':  [0.0, 0.5, 1.0],
    }
    thresholds = [(0.05, -0.05), (0.08, -0.08), (0.10, -0.10), (0.12, -0.12), (0.15, -0.15)]
    best = None
    keys = list(grid.keys())
    for vals in itertools.product(*[grid[k] for k in keys]):
        w = dict(zip(keys, vals))
        for buy_th, sell_th in thresholds:
            m = backtest_metrics(feats, w, buy_th, sell_th)
            obj = objective(m)
            if best is None or obj > best['obj']:
                best = {'obj': obj, 'weights': w, 'buy_th': buy_th, 'sell_th': sell_th, **m}
    return best

def grid_search_tsla(feats):
    grid = {
        'w_news':     [0.6, 0.8, 1.0, 1.2],
        'w_filing':   [0.0, 0.3, 0.6],
        'w_disagree': [0.0, 0.5, 1.0],
        'w_trend':    [0.0, 0.2, 0.4],
        'w_mom':      [0.5, 1.0, 1.5],
        'w_peer':     [0.0, 0.5, 1.0],
        'w_sector':   [0.0, 0.5, 1.0],
        'w_macro':    [0.0, 0.5],
    }
    thresholds = [(0.08, -0.08), (0.12, -0.12), (0.15, -0.15), (0.20, -0.20)]
    best = None
    keys = list(grid.keys())
    for vals in itertools.product(*[grid[k] for k in keys]):
        w = dict(zip(keys, vals))
        for buy_th, sell_th in thresholds:
            m = backtest_metrics(feats, w, buy_th, sell_th)
            obj = objective(m)
            if best is None or obj > best['obj']:
                best = {'obj': obj, 'weights': w, 'buy_th': buy_th, 'sell_th': sell_th, **m}
    return best

In [ ]:
best_btc = grid_search_btc(btc_feats)
print('BTC best:')
for k, v in best_btc.items():
    print(' ', k, v)

In [ ]:
best_tsla = grid_search_tsla(tsla_feats)
print('TSLA best:')
for k, v in best_tsla.items():
    print(' ', k, v)

## 10. Plot the best portfolios

In [ ]:
import matplotlib.pyplot as plt

def plot_best(feats, best, asset):
    f = feats.sort_values('date').reset_index(drop=True).copy()
    f['score']    = f.apply(lambda r: combine_score(r, best['weights'], asset), axis=1)
    f['position'] = actions_from_scores(f['score'].values, best['buy_th'], best['sell_th'])
    f['ret']      = f['price'].pct_change().fillna(0)
    f['pos_lag']  = f['position'].shift(1).fillna(0)
    f['strat_ret']= f['pos_lag'] * f['ret']
    f['cum_strat']  = (1 + f['strat_ret']).cumprod()
    f['cum_bh']     = (1 + f['ret']).cumprod()
    plt.figure(figsize=(11, 5))
    plt.plot(f['date'], f['cum_bh'],    label=f'{asset} buy-and-hold')
    plt.plot(f['date'], f['cum_strat'], label='Hybrid strategy')
    plt.legend(); plt.title(f'{asset} — best hybrid vs buy-and-hold')
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

plot_best(btc_feats, best_btc, 'BTC')
plot_best(tsla_feats, best_tsla, 'TSLA')

## 11. Print the params to drop into `app.py`

In [ ]:
import json
params = {
    'BTC':  {'weights': best_btc['weights'],  'buy_th': best_btc['buy_th'],  'sell_th': best_btc['sell_th']},
    'TSLA': {'weights': best_tsla['weights'], 'buy_th': best_tsla['buy_th'], 'sell_th': best_tsla['sell_th']},
}
with open('tuned_params.json', 'w') as f:
    json.dump(params, f, indent=2)
print(json.dumps(params, indent=2))
print('\nSaved -> tuned_params.json')

## 12. Send `tuned_params.json` back

Once this finishes, hand me `tuned_params.json` (or paste its contents). I will fold those values into `app.py`, redeploy to Fly, and we move to URL submission.